# Stage 4 v2 — BERTopic on Filtered Corpus

Improvements over v1:
- Input is `chunks_filtered.parquet` — off-topic r/singapore content removed by `ns_filter.py`
- `min_cluster_size=200`, `n_neighbors=30`, `min_samples=20` — fewer, tighter clusters
- `calculate_probabilities=False` — avoids the OOM that killed v1
- Outlier rescue via BERTopic's `reduce_outliers()` API (two-pass: embeddings → c-TF-IDF)
  instead of raw cosine similarity; followed by `update_topics()` to sharpen representations
- Full corpus fit — no subsampling, no granularity loss

**Datasets needed:**
- `ns-sentiment-filtered` — `chunks_filtered.parquet` (output of `src/features/ns_filter.py`)

**Setup:** GPU T4 x2 · Save & Run All (committed mode — never run interactively)

In [ ]:
# Cell 1 — Discover exact input paths
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 2 — Config  ← UPDATE PATHS AFTER RUNNING CELL 1
CHUNKS_FILTERED = '/kaggle/input/<your-dataset>/chunks_filtered.parquet'
OUT_DIR         = '/kaggle/working'

UMAP_N_COMPONENTS = 5
UMAP_N_NEIGHBORS  = 30     # wider than v1 (15) — better global structure
UMAP_MIN_DIST     = 0.0
UMAP_METRIC       = 'cosine'
HDBSCAN_MIN_SIZE  = 200    # larger than v1 (50) — kills micro-fragments
HDBSCAN_MIN_SAMPLES = 20   # stricter core point definition
RANDOM_STATE      = 42
NR_COARSE_TOPICS  = 20

# Outlier rescue thresholds
RESCUE_EMB_THRESHOLD   = 0.3   # pass 1: cosine similarity to topic centroid
RESCUE_CTFIDF_THRESHOLD = 0.1  # pass 2: c-TF-IDF token overlap score

_SINGLISH_STOP = ['lah', 'lor', 'leh', 'sia', 'meh', 'hor', 'wah', 'ah']

In [ ]:
# Cell 3 — Install + imports
!pip install bertopic safetensors sentence-transformers -q

import numpy as np
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from cuml.cluster import HDBSCAN
from cuml.manifold import UMAP
from sklearn.feature_extraction.text import CountVectorizer

print('Imports OK')

In [ ]:
# Cell 4 — Load filtered chunks and stack embeddings
df = pd.read_parquet(CHUNKS_FILTERED)
print(f'Filtered chunks: {len(df):,}')
print(f'Subreddit breakdown:\n{df["subreddit"].value_counts().to_string()}')

first = df['embedding'].iloc[0]
embeddings = (
    np.stack(df['embedding'].values).astype(np.float32)
    if isinstance(first, np.ndarray)
    else np.array(df['embedding'].tolist(), dtype=np.float32)
)
print(f'Embedding matrix: {embeddings.shape}')
docs = df['text'].tolist()

In [ ]:
# Cell 5 — Build BERTopic model
#
# calculate_probabilities=False: avoids (n_docs × n_topics) dense matrix that
# OOM-killed v1. Has zero effect on cluster quality — only affects whether
# HDBSCAN computes soft membership scores post-clustering.
# Outlier rescue is handled via reduce_outliers() instead.

base_stop = CountVectorizer(stop_words='english').get_stop_words()
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=list(base_stop) + _SINGLISH_STOP,
    min_df=10,
)
representation_model = [
    KeyBERTInspired(),
    MaximalMarginalRelevance(diversity=0.3),
]
umap_model = UMAP(
    n_components=UMAP_N_COMPONENTS,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric='euclidean',
    prediction_data=True,
)
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    representation_model=representation_model,
    embedding_model='sentence-transformers/all-mpnet-base-v2',
    calculate_probabilities=False,
    verbose=True,
)

In [ ]:
# Cell 6 — Fit BERTopic on full filtered corpus
topics_raw, probs_raw = topic_model.fit_transform(docs, embeddings)
topics_raw = np.array(topics_raw, dtype=np.int16)

# probs_raw may be None or zeros with calculate_probabilities=False
# Initialise: 1.0 for clustered points, 0.0 for outliers
if probs_raw is None or (hasattr(probs_raw, '__len__') and len(probs_raw) == 0):
    probs_raw = np.where(topics_raw != -1, 1.0, 0.0).astype(np.float32)
else:
    probs_raw = np.array(probs_raw, dtype=np.float32)
    if probs_raw.ndim > 1:
        probs_raw = probs_raw.max(axis=1)
    probs_raw = np.where(topics_raw != -1, np.maximum(probs_raw, 1e-6), 0.0).astype(np.float32)

n_fine    = len(set(topics_raw)) - (1 if -1 in topics_raw else 0)
n_outlier = (topics_raw == -1).sum()
print(f'Fine topics: {n_fine}  |  Initial outliers: {n_outlier:,} ({n_outlier / len(topics_raw) * 100:.1f}%)')

In [ ]:
# Cell 7 — Save fine model and keyword table
# Must happen BEFORE reduce_topics() which modifies the model in-place.
topic_model.save(f'{OUT_DIR}/bertopic_fine',
                 serialization='safetensors', save_ctfidf=True)
print('Saved bertopic_fine')

def export_keywords(model, label):
    rows = []
    for _, row in model.get_topic_info().iterrows():
        tid = row['Topic']
        if tid == -1:
            continue
        kws = model.get_topic(tid)
        rows.append({
            'topic_id': tid,
            'count':    row['Count'],
            'name':     row.get('Name', ''),
            'keywords': ', '.join(w for w, _ in kws[:10]),
            'scores':   ', '.join(f'{s:.4f}' for _, s in kws[:10]),
        })
    pd.DataFrame(rows).to_csv(f'{OUT_DIR}/topic_keywords_{label}.csv', index=False)
    print(f'Saved topic_keywords_{label}.csv  ({len(rows)} topics)')

export_keywords(topic_model, 'fine')

In [ ]:
# Cell 8 — Hierarchical topic tree
hierarchical_df = topic_model.hierarchical_topics(docs)
hierarchical_df.to_parquet(f'{OUT_DIR}/hierarchical_topics.parquet', index=False)
print('Saved hierarchical_topics.parquet')

In [ ]:
# Cell 9 — Two-pass outlier rescue via reduce_outliers()
#
# Pass 1 — embeddings strategy:
#   Assigns each outlier to the topic whose centroid embedding has the highest
#   cosine similarity, if that similarity >= RESCUE_EMB_THRESHOLD.
#   Fast, uses topic_model.topic_embeddings_ (pre-computed centroids).
#
# Pass 2 — c-TF-IDF strategy:
#   For chunks still unassigned after pass 1, scores token overlap between
#   the chunk's vocabulary and each topic's c-TF-IDF keyword distribution.
#   More precise for text: a chunk mentioning 'IPPT', 'recruit', 'sergeant'
#   will score strongly against the relevant NS topic via keyword matching
#   even if its embedding sits ambiguously between clusters.
#
# update_topics() recalculates c-TF-IDF with rescued outliers now counted
# as topic members — sharpens keyword representations before coarse reduction.

n_before = (topics_raw == -1).sum()
print(f'Outliers before rescue: {n_before:,}')

# Pass 1: embedding similarity
topics_pass1 = topic_model.reduce_outliers(
    docs,
    topics_raw.tolist(),
    strategy='embeddings',
    threshold=RESCUE_EMB_THRESHOLD,
    embeddings=embeddings,
)
topics_pass1 = np.array(topics_pass1, dtype=np.int16)
n_after_pass1 = (topics_pass1 == -1).sum()
print(f'After pass 1 (embeddings): {n_after_pass1:,} outliers remain  '
      f'({n_before - n_after_pass1:,} rescued)')

# Pass 2: c-TF-IDF token overlap
topics_pass2 = topic_model.reduce_outliers(
    docs,
    topics_pass1.tolist(),
    strategy='c-TF-IDF',
    threshold=RESCUE_CTFIDF_THRESHOLD,
)
topics_fine = np.array(topics_pass2, dtype=np.int16)
n_after_pass2 = (topics_fine == -1).sum()
print(f'After pass 2 (c-TF-IDF):  {n_after_pass2:,} outliers remain  '
      f'({n_after_pass1 - n_after_pass2:,} rescued)')
print(f'Total rescued: {n_before - n_after_pass2:,} / {n_before:,}  '
      f'({(n_before - n_after_pass2) / max(n_before, 1) * 100:.1f}%)')

# Build prob array: 1.0 for original assignments, 0.5 for rescued
probs_fine = probs_raw.copy()
newly_rescued = (topics_raw == -1) & (topics_fine != -1)
probs_fine[newly_rescued] = 0.5

# Incorporate rescued outliers into topic representations
print('Updating topic representations with rescued outliers …')
topic_model.update_topics(docs, topics=topics_fine.tolist())
print('update_topics() done')

# Re-export fine keywords now that representations are updated
export_keywords(topic_model, 'fine')
topic_model.save(f'{OUT_DIR}/bertopic_fine',
                 serialization='safetensors', save_ctfidf=True)
print('Resaved bertopic_fine with updated representations')

In [ ]:
# Cell 10 — Coarse reduction to NR_COARSE_TOPICS
#
# reduce_topics() cuts the hierarchical dendrogram and modifies the model
# in-place. In newer BERTopic versions it returns the model (not a tuple).
# Retrieve updated assignments via topic_model.topics_.
#
# Rescued outliers are invisible to reduce_topics (they're still -1 in
# topics_raw), so we derive their coarse topic from an empirical
# fine→coarse lookup built from non-outlier docs.

print(f'Reducing to {NR_COARSE_TOPICS} coarse topics …')
topic_model.reduce_topics(docs, nr_topics=NR_COARSE_TOPICS)
topics_coarse_raw = np.array(topic_model.topics_, dtype=np.int16)

# Build fine→coarse lookup from non-outlier docs
fine_to_coarse = {}
for fine, coarse in zip(topics_raw, topics_coarse_raw):
    if fine != -1 and coarse != -1 and fine not in fine_to_coarse:
        fine_to_coarse[int(fine)] = int(coarse)

topics_coarse = np.array(
    [fine_to_coarse.get(int(t), t) if t != -1 else -1 for t in topics_fine],
    dtype=np.int16,
)
print(f'Coarse unmapped: {(topics_coarse == -1).sum():,}')

topic_model.save(f'{OUT_DIR}/bertopic_coarse',
                 serialization='safetensors', save_ctfidf=True)
export_keywords(topic_model, 'coarse')

In [ ]:
# Cell 11 — Build assignments table and write back to chunks_filtered
assignments = pd.DataFrame({
    'chunk_id':        df['chunk_id'].values,
    'doc_id':          df['doc_id'].values,
    'doc_type':        df['doc_type'].values,
    'subreddit':       df['subreddit'].values,
    'created_utc':     df['created_utc'].values,
    'topic_id_fine':   topics_fine,
    'topic_prob_fine': probs_fine,
    'topic_id_coarse': topics_coarse,
})
assignments.to_parquet(f'{OUT_DIR}/chunk_topics.parquet', index=False)
print(f'Saved chunk_topics.parquet  ({len(assignments):,} rows)')

# Write topic columns back into chunks_filtered
merge_cols = ['chunk_id', 'topic_id_fine', 'topic_prob_fine', 'topic_id_coarse']
out_df = df.drop(columns=[c for c in merge_cols[1:] if c in df.columns])
out_df = out_df.merge(assignments[merge_cols], on='chunk_id', how='left')
out_df.to_parquet(f'{OUT_DIR}/chunks_filtered.parquet', index=False)
print(f'Saved chunks_filtered.parquet with updated topic columns')

# Quality summary
n_fine_topics   = int((topics_fine  != -1).sum())
n_coarse_topics = int((topics_coarse != -1).sum())
outlier_rate    = (topics_fine == -1).mean() * 100
print(f'\n── Quality summary ──')
print(f'Total chunks:      {len(df):,}')
print(f'Assigned (fine):   {n_fine_topics:,}  ({100 - outlier_rate:.1f}%)')
print(f'Outliers (fine):   {(topics_fine == -1).sum():,}  ({outlier_rate:.1f}%)')
print(f'Fine topics:       {len(set(topics_fine)) - (1 if -1 in topics_fine else 0)}')
print(f'Coarse topics:     {NR_COARSE_TOPICS}')
print('\nStage 4 v2 complete.')